# COMP5329 — Deep Learning

**Tutorial — Self-Supervised Representation Learning**

**Semester 1, 2026**

### Learning Objectives
By the end of this tutorial you will be able to:
1. Explain **why** self-supervised learning (SSL) is needed and place the major methods into a taxonomy (pretext / generative-masked / contrastive / non-contrastive).
2. Describe the **masked-modelling paradigm** and implement the two defining pieces of a Masked Autoencoder (MAE): random patch masking and the masked reconstruction loss.
3. Describe the **contrastive paradigm** and implement SimCLR's NT-Xent (InfoNCE) loss from scratch — similarity matrix, diagonal masking, temperature, cross-entropy against positive-pair indices.
4. Reason about **temperature** in contrastive losses and about **mask ratio** in masked modelling.
5. Answer exam-style short-answer questions comparing the two paradigms, including the role of negatives (and why SimSiam can do without them).

### Topic Coverage

Week 11 covers **Self-Supervised Representation Learning**. The full topic list (see `Week11_Self_Study.ipynb`) is:

- ✅ **Motivation & taxonomy of SSL** — label-free learning, pretext tasks, generative vs contrastive vs non-contrastive *(tutorial)*
- ✅ **Masked modelling — MAE** — random masking, asymmetric encoder/decoder, loss on masked positions only *(tutorial)*
- ✅ **Contrastive learning — SimCLR / NT-Xent (InfoNCE)** — positive pairs, negatives, temperature *(tutorial)*
- 📖 **Autoencoder / bottleneck reconstruction** and its connection to PCA *(self-study)*
- 📖 **MoCo** — momentum encoder + queue of negatives *(self-study)*
- 📖 **Non-contrastive methods** — BYOL / SimSiam (stop-gradient) and DINO (self-distillation + centering) *(self-study)*
- 📖 **CLIP** — cross-modal InfoNCE between image and text *(self-study)*

To keep the live session focused, the tutorial covers **exactly two canonical representatives** — one from each major paradigm — thoroughly enough that the code is exam-ready. Everything else lives in the self-study notebook.

The live session is organised into three parts: **Part A** — tutor walkthrough, **Part B** — in-class coding exercise, **Part C** — exam-style Q&A.

---
# Part A · Tutor Review

> **Goal.** By the end of Part A you should be able to (a) state the core idea of SSL in one sentence, (b) place any SSL method you meet into one of four boxes on the taxonomy, and (c) explain why MAE and SimCLR are the canonical representatives of the two dominant paradigms you will implement in Part B.

---
## §0 — Why Self-Supervised Learning?

Modern deep models need **lots of data**. Collecting *labels* for that data is the expensive part: ImageNet took years of crowd-sourcing, and medical imaging datasets can cost millions of dollars in expert annotation.

**Self-supervised learning** sidesteps this bottleneck: it generates its own supervision signal from the *structure* of the unlabelled data itself.

- **Supervised**: needs $(x, y)$ pairs — a human must provide $y$.
- **Unsupervised** (classical): clustering / density estimation, no downstream task in mind.
- **Self-supervised**: define a **pretext task** whose label can be computed automatically from $x$ alone, train on that, and then **transfer** (fine-tune or linear-probe) the learned representation to the real downstream task.

This "pre-train on cheap data, fine-tune on expensive labels" recipe is what powers modern foundation models (BERT, GPT, MAE, CLIP, DINOv2, …).

---
## §1 — Taxonomy of SSL Methods

Every SSL method defines a **pretext task** plus a **loss**. Four broad families:

| Family | Pretext task | Loss | Representative |
|---|---|---|---|
| **Classical pretext** | Predict rotation / jigsaw / colourisation | Task-specific CE / L2 | RotNet, Jigsaw |
| **Generative / masked** | Reconstruct the input (possibly after hiding part of it) | Pixel / token MSE or CE | **AE, MAE**, BERT, GPT |
| **Contrastive** | Make two views of the same input agree, push different inputs apart | InfoNCE / NT-Xent | **SimCLR**, MoCo, CLIP |
| **Non-contrastive** | Make two views agree, avoid collapse without explicit negatives | MSE + stop-gradient / centering | BYOL, SimSiam, DINO |

The **two paradigms that dominate 2020–2025 literature** are:

1. **Masked modelling** — *"hide part of the input, predict it."* Works brilliantly for language (BERT) and, with the right recipe, for images (MAE).
2. **Contrastive learning** — *"two views of the same thing should land in the same spot in embedding space."*

These are the two methods you will implement in Part B. The other families are important but are covered in the self-study notebook.

---
## §2 — A Whirlwind Tour of the Self-Study Methods

These methods are **not coded in this tutorial** — one line each, just enough context so that when you see them referenced in Part C or in a paper, you know where they sit.

- **Autoencoder (AE)** — classic bottleneck reconstruction $x \to z \to \hat{x}$, trained with MSE. A *linear* AE with tied weights and MSE loss provably learns the principal components of the data (it spans the same subspace as PCA). MAE generalises this idea by *masking* part of the input instead of relying on a low-dim bottleneck.
- **BYOL / SimSiam** — two-branch Siamese networks that avoid representation collapse **without negatives**, relying on a **stop-gradient** on one branch plus an asymmetric predictor head. SimSiam's one-line recipe is: `loss = - cos_sim(predictor(z_a), stop_grad(z_b))`.
- **DINO** — self-distillation: a student network is trained to match a *teacher* network (EMA of the student), with the teacher's outputs **centered** and **sharpened** to prevent collapse.
- **CLIP** — contrastive learning across **modalities**: image encoder and text encoder are trained with symmetric InfoNCE so that each image embedding matches its own caption's embedding more than any other caption in the batch.

Each of the above can be understood as a variation on the two paradigms you are about to implement: MAE (masked modelling) and SimCLR (contrastive learning).

---
## §3 — Paradigm 1: Masked Autoencoder (MAE)

**Core idea.** Take an image, split it into non-overlapping patches (à la Vision Transformer, Week 8), **hide 75%** of the patches at random, and train a model to reconstruct the missing pixels.

$$\mathcal{L}_{\text{MAE}} = \frac{1}{|\mathcal{M}|}\sum_{i \in \mathcal{M}} \lVert \hat{x}_i - x_i \rVert_2^2$$

where $\mathcal{M}$ is the set of **masked** patch indices. Three design choices are responsible for MAE's success:

1. **A high mask ratio — 75%.** BERT uses 15% because language is redundant at the sub-word level but not at the paragraph level. Images are **far more redundant**: neighbouring patches share colour, texture, and edges, so at 15% masking the model can just interpolate locally. Raising the ratio to 75% makes the task genuinely non-trivial and forces the encoder to learn *semantic* structure.
2. **An asymmetric encoder / decoder.** The encoder processes **only the visible (25%) patches** — this is a 4× speedup vs encoding the full image — and the small decoder is the only place where masked positions (represented by a shared `[mask]` token) are processed. The decoder is discarded after pre-training; only the encoder transfers.
3. **Loss computed only on masked positions.** If you included the visible patches in the loss, the model could learn a trivial identity shortcut on them. Restricting the loss to $\mathcal{M}$ forces every gradient to come from a *reconstruction-from-context* signal.

In Part B you will implement the two pieces that encode these choices:

- `random_masking(patches, mask_ratio)` — randomly select which patches to hide, return the visible subset, the binary mask, and the indices needed to restore the original order.
- `mae_loss(pred_patches, target_patches, mask)` — MSE **restricted to masked positions**.

We will *not* train a ViT encoder — that is too heavy for a tutorial. The demo uses a synthetic patch tensor so we can focus on the masking and loss logic.

---
## §4 — Paradigm 2: SimCLR / NT-Xent

**Core idea.** Take a mini-batch of $B$ images. For each image $x_i$ produce **two augmented views** $\tilde{x}_i$ and $\tilde{x}_i'$ (random crop + colour jitter + …). Push both views through the same encoder $f$ + projection head $g$ to obtain L2-normalised embeddings $z_i, z_i' \in \mathbb{R}^D$. Now treat the $2B$ embeddings as a classification problem:

> For each query $z_i$, the **positive** is its partner view $z_i'$ and the **negatives** are the other $2B-2$ embeddings in the batch.

The **NT-Xent** (Normalised Temperature-scaled Cross Entropy) loss for a positive pair $(i, j)$ is

$$\ell_{i,j} = -\log \frac{\exp(\text{sim}(z_i, z_j) / \tau)}{\sum_{k=1}^{2B} \mathbb{1}_{k \ne i}\,\exp(\text{sim}(z_i, z_k) / \tau)}$$

and the full loss is the average over all $2B$ positive pairs. Three design choices matter:

1. **Cosine similarity on L2-normalised embeddings.** Similarity is $\text{sim}(z_i, z_j) = z_i^\top z_j / (\lVert z_i\rVert\lVert z_j\rVert)$. Normalising first means the $\lVert \cdot\rVert$ factors are 1, so the dot product **is** the cosine, and the loss is invariant to overall embedding magnitude — only *directions* are compared.
2. **Mask out self-similarities.** The diagonal of the $(2B, 2B)$ similarity matrix is always 1 (each embedding compared to itself) and would dominate the softmax denominator. We **remove the diagonal** before computing cross-entropy.
3. **Temperature $\tau$.** A small $\tau$ sharpens the softmax (penalises hard negatives heavily); a large $\tau$ flattens it (all negatives look similar). Typical values: SimCLR $\tau \approx 0.1$–$0.5$, CLIP $\tau \approx 0.07$.

In Part B you will implement `nt_xent_loss(z1, z2, temperature)` where `z1, z2 ∈ (B, D)` are the already-normalised embeddings of the two views.

---
## §5 — Masked vs Contrastive: Where They Differ

| Aspect | Masked (MAE) | Contrastive (SimCLR) |
|---|---|---|
| **Loss target** | Pixels of masked patches | Cosine similarity of projected embeddings |
| **Augmentations** | Just masking — no colour / crop tricks | Aggressive: random crop, colour jitter, blur, … |
| **Batch size** | Small OK (64–256) | **Large is critical** (1024–8192) — negatives come from the batch |
| **Representation quality** | Excellent for fine-tuning, weaker for linear-probe | Excellent for linear-probe, slightly weaker for full fine-tuning |
| **Intuition** | Learn low-level *and* semantic features by reconstructing context | Learn invariances dictated by the augmentation set |

These two recipes are the foundations you need. Now let us code them.

---
# Part B · In-Class Exercise

> **Your job**: fill in the `# TODO` blocks in the two tasks below. Each task has a collapsed **Solution** cell underneath — try the task yourself first, then expand the solution to compare.
>
> There are **two tasks** — MAE masking + loss, then SimCLR's NT-Xent loss. Both are pure loss-computation demos on synthetic tensors (no training loops) so you can focus on the core logic.

In [ ]:
import math
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)

### Task B1 · MAE — random masking + masked reconstruction loss

You are given a **patch tensor** `patches` of shape $(B, N, D)$, where $B$ is the batch size, $N$ is the number of patches per image, and $D$ is the patch embedding dimension. (In a full ViT-MAE you would produce this by linearly projecting non-overlapping image patches; here we skip that step and work with synthetic tensors.)

You will implement two functions:

1. `random_masking(patches, mask_ratio)` — for each image, randomly pick $\lfloor N \cdot \text{mask\_ratio} \rfloor$ patch positions to hide. Return:
    - `x_visible` — the **kept** patches, shape $(B, N_{\text{keep}}, D)$
    - `mask` — a binary tensor of shape $(B, N)$ with `1` at masked positions, `0` at visible positions
    - `ids_restore` — the permutation indices needed to put the (visible + mask-token) sequence back into the original patch order (you will need this in a real MAE decoder, and it is a common exam trap to compute correctly)

2. `mae_loss(pred, target, mask)` — MSE between `pred` and `target` computed **only at positions where `mask == 1`**. Both `pred` and `target` are $(B, N, D)$.

**Hint.** The standard MAE trick for `random_masking` is: generate a random noise tensor of shape $(B, N)$, `argsort` it to get a random permutation (`ids_shuffle`), then the first $N_{\text{keep}}$ indices of that permutation select the visible patches. A second `argsort` of `ids_shuffle` gives `ids_restore`.

In [ ]:
def random_masking(patches: torch.Tensor, mask_ratio: float):
    """Randomly mask patches per-example.

    Parameters
    ----------
    patches    : (B, N, D) tensor of patch embeddings.
    mask_ratio : fraction of patches to mask (e.g. 0.75).

    Returns
    -------
    x_visible   : (B, N_keep, D) — the kept patches, in shuffled order.
    mask        : (B, N)        — 1 = masked, 0 = visible (in ORIGINAL order).
    ids_restore : (B, N)        — permutation to undo the shuffle.
    """
    B, N, D = patches.shape
    n_keep = int(N * (1.0 - mask_ratio))

    # TODO 1 — generate random noise in [0,1) of shape (B, N), then argsort it
    #          along dim=1 to get `ids_shuffle`. The first n_keep columns of
    #          ids_shuffle are the indices of the patches we will KEEP.
    #          Also compute `ids_restore = argsort(ids_shuffle)` — applying
    #          ids_restore to any shuffled sequence undoes the shuffle.
    noise       = ...
    ids_shuffle = ...
    ids_restore = ...

    # Gather the visible patches. (Provided — study how torch.gather works here.)
    ids_keep  = ids_shuffle[:, :n_keep]                                 # (B, n_keep)
    x_visible = torch.gather(patches, dim=1,
                             index=ids_keep.unsqueeze(-1).expand(-1, -1, D))

    # TODO 2 — build the binary mask in ORIGINAL patch order.
    #          Construct a length-N vector that is 0 for the first n_keep
    #          positions and 1 for the rest, broadcast it to (B, N), and
    #          then unshuffle it with ids_restore so the mask is aligned
    #          with the original patch positions (not the shuffled order).
    mask = ...

    return x_visible, mask, ids_restore


def mae_loss(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Mean-squared error computed ONLY on masked positions.

    Parameters
    ----------
    pred   : (B, N, D) — reconstructed patches, in original order.
    target : (B, N, D) — ground-truth patches,  in original order.
    mask   : (B, N)    — 1 = masked (include in loss), 0 = visible (ignore).
    """
    # TODO 3 — per-patch MSE averaged over the feature dim D, giving (B, N).
    per_patch = ...

    # TODO 4 — average `per_patch` ONLY over positions where mask == 1.
    #          Use (per_patch * mask).sum() / mask.sum() to ignore visible
    #          patches entirely.
    loss = ...
    return loss


# ── Demo on a synthetic patch tensor ──────────────────────────────────────
B, N, D = 2, 16, 32
patches = torch.randn(B, N, D)

x_visible, mask, ids_restore = random_masking(patches, mask_ratio=0.75)
print(f"patches      : {tuple(patches.shape)}")
print(f"x_visible    : {tuple(x_visible.shape)}  (expected: (2, 4, 32) — 25% of 16)")
print(f"mask         : {tuple(mask.shape)}  (fraction masked = {mask.float().mean().item():.2f})")
print(f"ids_restore  : {tuple(ids_restore.shape)}")

# Loss demo: pretend our decoder output is zero everywhere.
pred = torch.zeros_like(patches)
loss = mae_loss(pred, patches, mask)
print(f"mae_loss (pred=0)    = {loss.item():.4f}")

# If pred == target exactly, loss must be 0 regardless of mask.
loss_zero = mae_loss(patches, patches, mask)
print(f"mae_loss (pred=target) = {loss_zero.item():.6f}")

<details>
<summary><b>▸ Solution · Task B1</b> (click to expand)</summary>

```python
def random_masking(patches, mask_ratio):
    B, N, D = patches.shape
    n_keep  = int(N * (1.0 - mask_ratio))

    # TODO 1 — random shuffle per example
    noise       = torch.rand(B, N, device=patches.device)
    ids_shuffle = torch.argsort(noise, dim=1)          # small → first
    ids_restore = torch.argsort(ids_shuffle, dim=1)    # inverse permutation

    ids_keep  = ids_shuffle[:, :n_keep]
    x_visible = torch.gather(patches, dim=1,
                             index=ids_keep.unsqueeze(-1).expand(-1, -1, D))

    # TODO 2 — mask in ORIGINAL order: 0 for visible, 1 for masked
    mask = torch.ones(B, N, device=patches.device)
    mask[:, :n_keep] = 0
    mask = torch.gather(mask, dim=1, index=ids_restore)   # un-shuffle

    return x_visible, mask, ids_restore


def mae_loss(pred, target, mask):
    per_patch = ((pred - target) ** 2).mean(dim=-1)        # (B, N)
    loss = (per_patch * mask).sum() / mask.sum().clamp(min=1.0)
    return loss
```

**Key points:**
- **`argsort` of random noise = random permutation.** This is the trick that lets you pick a *uniformly random subset* of size $N_{\text{keep}}$ per example in parallel, without a Python loop.
- **`ids_restore` is `argsort(ids_shuffle)`** — the inverse permutation. In a real MAE, after the encoder produces embeddings for the visible patches, you concatenate a learned `[mask]` token for each masked position and then `gather` with `ids_restore` to put everything back in the original patch order before feeding the decoder. Getting this index right is what the TODO exercises.
- **Loss on masked positions only.** The denominator is `mask.sum()`, **not** `mask.numel()` — we are averaging over masked patches only. If you accidentally divide by `numel()`, the loss scales with `(1 - mask_ratio)` and the gradient signal at each masked patch is attenuated.
- **Why not include visible patches?** If the model is given the visible patch on the input side *and* asked to reproduce it on the output side, it can solve that part of the objective with an identity shortcut — no learning happens there. Restricting the loss to $\mathcal{M}$ means every ounce of gradient comes from *predicting hidden content from visible context*, which is the only signal that forces the encoder to build a semantic representation.
- **Why 75%?** Lower ratios let the model solve masking by local interpolation — neighbouring patches share colour and texture. At 75% the task is genuinely hard, matching how vision needs a global, semantic model.
</details>

### Task B2 · SimCLR — NT-Xent from scratch

Given two batches of already-L2-normalised embeddings `z1, z2 ∈ (B, D)` (the two augmented views of the same $B$ images), implement `nt_xent_loss(z1, z2, temperature)`.

**Recipe.**

1. Stack `z1, z2` into `z ∈ (2B, D)`.
2. Compute the full similarity matrix `sim = (z @ z.T) / temperature` — shape $(2B, 2B)$.
3. Remove the diagonal (each row's self-similarity is always $1/\tau$ and would win every softmax). After masking, reshape to $(2B, 2B-1)$.
4. Build a `labels` vector that points at each row's positive pair in the masked matrix, and call `F.cross_entropy(sim, labels)`.

**Label indexing (the trap!).** After removing the diagonal, row $i$ has $2B-1$ columns. The positive for row $i$ lives at column index
- $B - 1 + i$ if $i \in [0, B)$ — row is a `z1` embedding, positive is the corresponding `z2`,
- $i - B$ if $i \in [B, 2B)$ — row is a `z2` embedding, positive is the corresponding `z1`.

Think about *why* those offsets are correct — the removal of the diagonal shifts every column index to the left by one for the positions after the removed cell.

In [ ]:
def nt_xent_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float = 0.5) -> torch.Tensor:
    """NT-Xent (normalised temperature-scaled cross-entropy) loss.

    Parameters
    ----------
    z1, z2      : (B, D) L2-normalised embeddings of two augmented views.
    temperature : scalar τ.
    """
    B, D = z1.shape

    # TODO 1 — build the (2B, 2B) similarity matrix scaled by 1/τ, then
    #          drop the diagonal so the result has shape (2B, 2B - 1).
    #          Use torch.cat + matmul, a boolean eye mask, and masked_select.
    z    = ...
    sim  = ...
    mask = ~torch.eye(2 * B, dtype=torch.bool)
    sim  = ...                           # shape (2B, 2B - 1)

    # TODO 2 — build the positive-pair labels and return cross-entropy.
    #   row i in [0, B)    → positive at column index (B - 1 + i)
    #   row i in [B, 2B)   → positive at column index (i - B)
    labels = ...
    return F.cross_entropy(sim, labels)


# ── Demo on a synthetic batch ─────────────────────────────────────────────
torch.manual_seed(0)
B, D = 8, 16

# Two random but L2-normalised batches — the "embedded views".
z1 = F.normalize(torch.randn(B, D), dim=1)
z2 = F.normalize(torch.randn(B, D), dim=1)

loss_random   = nt_xent_loss(z1, z2, temperature=0.5)
loss_identity = nt_xent_loss(z1, z1.clone(), temperature=0.5)
print(f"Loss on identical views (low, ideally near 0): {loss_identity.item():.4f}")
print(f"Loss on random   views (≈ log(2B-1) = {math.log(2*B-1):.4f}): {loss_random.item():.4f}")

# ── Temperature sweep: how does τ affect the loss on random embeddings? ────
taus  = [0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0]
vals  = [nt_xent_loss(z1, z2, temperature=t).item() for t in taus]
plt.figure(figsize=(6, 4))
plt.semilogx(taus, vals, marker='o')
plt.axhline(math.log(2 * B - 1), linestyle='--', alpha=0.5,
            label='log(2B−1) = uniform softmax')
plt.xlabel('temperature τ'); plt.ylabel('NT-Xent loss')
plt.title('Effect of τ on NT-Xent (random views)')
plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

print("τ → 0 : softmax becomes one-hot on the hardest negative → noisy, unstable.")
print("τ → ∞ : softmax becomes uniform → loss saturates at log(2B-1) → no signal.")

<details>
<summary><b>▸ Solution · Task B2</b> (click to expand)</summary>

```python
def nt_xent_loss(z1, z2, temperature=0.5):
    B = z1.size(0)

    # TODO 1 — similarity matrix with diagonal removed
    z    = torch.cat([z1, z2], dim=0)                    # (2B, D)
    sim  = (z @ z.T) / temperature                       # (2B, 2B)
    mask = ~torch.eye(2 * B, dtype=torch.bool)           # drop diagonal
    sim  = sim.masked_select(mask).view(2 * B, 2 * B - 1)

    # TODO 2 — positive-pair labels + cross-entropy
    labels = torch.cat([
        torch.arange(B - 1, 2 * B - 1),   # rows 0..B-1: positive = B-1+i
        torch.arange(0, B),               # rows B..2B-1: positive = i - B
    ])
    return F.cross_entropy(sim, labels)
```

**Key points:**
- **NT-Xent is cross-entropy, not a new loss.** Once you have the $(2B, 2B-1)$ logit matrix and a label vector pointing at each row's positive, the whole thing reduces to a standard $(2B-1)$-way classification. The "N" in NT-Xent just means the embeddings are L2-**N**ormalised first; the "T" is the **T**emperature scaling.
- **Why L2-normalise the embeddings?** With $\lVert z_i\rVert = 1$, the dot product $z_i^\top z_j$ **is** the cosine similarity, so $\text{sim} \in [-1, 1]$ and the loss depends only on angles, not magnitudes. Without normalisation, the model could minimise the loss by scaling all positive embeddings to have huge norm — a degenerate shortcut that has nothing to do with representation quality.
- **Why drop the diagonal?** The diagonal of `z @ z.T / τ` is always `1/τ` (each embedding dotted with itself), which is the largest possible entry and would dominate the softmax denominator of **every** row. Removing it is equivalent to the $\mathbb{1}_{k \ne i}$ indicator in the formal NT-Xent formula.
- **Label indexing after the diagonal drop.** In the un-masked matrix, the positive for row $i < B$ lives at column $B + i$ (its partner $z2$ in the stacked matrix). After we delete the diagonal, every column index to the *right* of $i$ shifts left by 1, so $B + i$ becomes $B + i - 1 = B - 1 + i$. Symmetric reasoning for rows $i \ge B$: positive was at column $i - B$, which is to the *left* of $i$, so no shift — the label is just `i - B`.
- **Temperature.** As $\tau \to 0$, the softmax concentrates on the single hardest negative and gradients become extreme and noisy. As $\tau \to \infty$, the softmax flattens to uniform and the loss saturates at $\log(2B-1)$ — no learning signal. The practical sweet spot for SimCLR is $\tau \approx 0.1$–$0.5$, for CLIP $\tau \approx 0.07$ (and CLIP actually *learns* it).
</details>

---
# Part C · Exam-Style Questions

> Three short-answer questions at medium-high difficulty. Each has a hidden **Model Answer** cell directly below with the key points the tutor will draw out during discussion.

## Q1 · MAE — why the loss is on masked positions only

In your Task B1 implementation, `mae_loss` returns `(per_patch * mask).sum() / mask.sum()`, i.e. the MSE is averaged **only** over patches where `mask == 1` (the hidden patches). A student suggests a "simpler" alternative: average the MSE over **all** $N$ patches (visible *and* masked), arguing it should still push the reconstructions to be accurate.

**(a)** Write the one-line code change that implements the student's alternative, and express the resulting loss as a weighted combination of the "visible" and "masked" terms. (Use $\mathcal{M}$ for the masked set and $\mathcal{V}$ for the visible set; $|\mathcal{M}| + |\mathcal{V}| = N$.)

**(b)** Why is the student's alternative a **bad** objective for representation learning? Your answer should mention the word "shortcut" and should reason about what a trained MAE encoder would do to minimise each of the two terms you identified in (a).

**(c)** MAE uses a **75%** mask ratio, whereas BERT (masked language modelling) uses only **15%**. Give one information-theoretic reason why the image-modality ratio needs to be so much higher.

<details>
<summary><b>▸ Model Answer · Q1</b></summary>

**(a)** The one-line change is replacing the masked average with a full mean:

```python
loss_all = ((pred - target) ** 2).mean()         # average over B * N * D
```

Equivalently,
$$\mathcal{L}_{\text{all}} = \frac{|\mathcal{V}|}{N}\underbrace{\frac{1}{|\mathcal{V}|}\sum_{i \in \mathcal{V}}\lVert\hat{x}_i - x_i\rVert^2}_{\text{visible term}} + \frac{|\mathcal{M}|}{N}\underbrace{\frac{1}{|\mathcal{M}|}\sum_{i \in \mathcal{M}}\lVert\hat{x}_i - x_i\rVert^2}_{\text{masked term}}.$$

At 75% masking, $|\mathcal{V}|/N = 0.25$ and $|\mathcal{M}|/N = 0.75$, so the visible term receives a quarter of the total weight.

**(b)** The visible term is trivially minimisable: the encoder sees patch $i \in \mathcal{V}$ as input and is asked to reproduce it as output. The model can route the visible patch through the network as an **identity shortcut** (a skip connection, or simply a near-identity transform in the decoder) and drive the visible-term MSE to ≈0 *without ever learning anything semantic*. Because $|\mathcal{V}|/N = 0.25$ is a large slice of the loss, optimisation will greedily exploit this shortcut, which in turn reduces the gradient pressure on the masked term. The encoder ends up as a glorified autoencoder-with-a-shortcut, and the representation transfers poorly to downstream tasks. Restricting the loss to $\mathcal{M}$ makes the shortcut impossible — the model can *only* receive gradient for predicting content it has **not** seen, which is exactly the signal that forces it to build a world-model of the input distribution.

**(c)** Text is **discrete and information-dense**: each word carries a lot of entropy, and with 15% masking BERT cannot simply copy a neighbour because adjacent words are usually semantically different. Images are **continuous and spatially redundant**: neighbouring patches share colour, texture, and edges, so with 15% masking a trivial linear interpolation of the visible neighbours already reconstructs the missing patch very well — the pretext task is too easy to force the model to learn global structure. Raising the mask ratio to 75% removes enough *local* context that the model *must* rely on **global, semantic** reasoning to fill in the missing regions, which is exactly the kind of representation we want to transfer.

**Key points:**
- Loss on masked-only = shortcut-free gradient = genuine representation learning.
- The mask ratio has to match the intrinsic redundancy of the modality — vision is redundant, language is not.
- The same argument explains why MAE needs an asymmetric encoder/decoder: running the big encoder on 25% of patches (vs 100%) is a free ~4× speedup that only works because the loss *ignores* the visible patches anyway.
</details>

## Q2 · SimCLR — temperature extremes

In Task B2 you plotted the NT-Xent loss as a function of $\tau$ on random (un-trained) embeddings and observed two failure regimes.

**(a)** For $\tau \to 0^{+}$: describe what the softmax distribution $p_k = \text{softmax}(\text{sim}_{i,k}/\tau)_k$ converges to, and what the gradient $\partial \mathcal{L} / \partial z_i$ concentrates on. Why is this a *training* problem rather than just a numerical one?

**(b)** For $\tau \to \infty$: describe the same two quantities, and compute the limiting loss value (as a closed-form expression in $B$). What does this imply for the gradient signal and therefore for learning?

**(c)** Production SimCLR uses $\tau \approx 0.1$–$0.5$ while CLIP uses $\tau \approx 0.07$ and additionally **learns** $\tau$ as a parameter (clamped so that $\log(1/\tau) \le \log 100$). Why does CLIP get away with a sharper temperature, and why does letting $\tau$ be a learned parameter act as a **self-regulation** mechanism?

<details>
<summary><b>▸ Model Answer · Q2</b></summary>

**(a)** As $\tau \to 0^{+}$, every logit $\ell_k = \text{sim}_{i,k}/\tau$ diverges in magnitude, and the softmax converges to a one-hot distribution on whichever non-self index has the largest similarity — i.e. the **single hardest negative** (or, once training is well underway, the positive). The loss gradient
$$\frac{\partial \mathcal{L}}{\partial z_i} = \frac{1}{\tau}\Big(\sum_{k} p_k z_k - z_{i^+}\Big)$$
has (i) a $1/\tau$ scaling that blows up, and (ii) a softmax weight vector $p$ concentrated on a *single* $k$. So each update step is a large, high-variance move that only accounts for one negative at a time; the next step may see a different hardest negative, and the representation oscillates. This is a **training** failure, not just numerical — even in infinite precision, the optimiser cannot average out gradient information from the other negatives, so convergence stalls or diverges.

**(b)** As $\tau \to \infty$, every logit $\ell_k \to 0$, so the softmax becomes the **uniform** distribution $p_k = 1/(2B-1)$ over the $2B-1$ non-self entries. The loss per row is therefore $-\log(1/(2B-1)) = \log(2B-1)$, independent of the actual embeddings — this is the dashed line in your temperature-sweep plot. The gradient
$$\frac{\partial \mathcal{L}}{\partial z_i} = \frac{1}{\tau}\Big(\sum_{k \ne i} \tfrac{1}{2B-1} z_k - z_{i^+}\Big) \to 0$$
because of the prefactor $1/\tau$. There is essentially no learning signal: the loss does not care which embedding is the positive, so the encoder cannot improve.

**(c)** CLIP's positive pair is (image, matching caption) — a **cross-modal** pair whose semantic gap to a random non-matching caption is enormous ("a dog on a beach" vs "a stock price chart"). The margin between positives and negatives in embedding space is intrinsically wide, so a sharper temperature can be tolerated without the $\tau \to 0$ failure mode. SimCLR's positives are two augmentations of the **same** image, and within a batch of natural images there are usually other images that are visually similar (same class, similar colour palette), so the positives-vs-negatives margin is narrower and $\tau$ must be larger to prevent hard negatives from producing noisy one-hot softmaxes. **Learnable $\tau$ self-regulates** because the cross-entropy loss, viewed as a function of $\tau$, is convex near the optimal sharpness: if training accidentally drives $\tau$ too small, the loss starts rising (we are in the noisy $\tau \to 0$ regime), and gradient descent on $\tau$ *increases* it again. The $\log(1/\tau) \le \log 100$ clamp is just a safety rail to keep $\tau$ from numerically underflowing.

**Key points:**
- Both extremes kill learning, for opposite reasons: $\tau\to 0$ has enormous but noisy gradients; $\tau\to\infty$ has vanishing gradients.
- The useful temperature depends on the **intrinsic positive/negative margin** of your task.
- Learning $\tau$ is a cheap trick that lets the model find its own sweet spot — you almost always want to do it.
</details>

## Q3 · Do we actually need negatives?  *(cross-cutting question)*

SimCLR's whole loss is built around contrasting a positive against $2B-2$ negatives. **SimSiam** (a non-contrastive method from the self-study notebook) uses *no negatives at all* — its loss is simply
$$\mathcal{L}_{\text{SimSiam}} = -\tfrac{1}{2}\big[\cos(\text{pred}(z_a),\, \text{stopgrad}(z_b)) + \cos(\text{pred}(z_b),\, \text{stopgrad}(z_a))\big]$$
and yet it learns useful representations comparable to SimCLR.

**(a)** What **failure mode** do negatives prevent in contrastive learning? Describe it concretely in terms of what the encoder might output in their absence.

**(b)** In SimSiam, negatives are absent — so what is preventing that failure mode? Explain the role played by (i) the **predictor head** on one branch only and (ii) the **stop-gradient** on the other branch. A one-sentence intuition per mechanism is enough, but both must appear.

**(c)** SimCLR's loss, in the limit of very large $B$, approximates a mutual-information lower bound between the two views; SimSiam has no such interpretation. Despite that, SimSiam often matches SimCLR's linear-probe accuracy on ImageNet. What does that tell you about the **role of negatives** in practice — are they about *information*, or about *optimisation*?

<details>
<summary><b>▸ Model Answer · Q3</b></summary>

**(a)** The failure mode is **representation collapse**: without negatives, the trivial solution to "make two views of the same image agree in embedding space" is for the encoder to map *every* input to the *same* constant vector. If $f(x) = c$ for all $x$, then the cosine similarity of any two views is 1 and the loss is 0 — a global minimum that is useless for downstream tasks. In practice the encoder doesn't collapse all the way to a point but to a **low-dimensional degenerate subspace**, which still destroys discriminative power. Negatives prevent this because pushing the constant-solution embedding apart from $2B-2$ "negatives" that are *also* equal to $c$ is infeasible — the softmax denominator in NT-Xent is saturated and the loss stays high, so the optimiser is forced to spread embeddings out.

**(b)**
- **(i) The predictor head**: SimSiam's online branch has an extra MLP $h$ on top of the projection, so that branch outputs $h(z_a)$ while the other outputs $z_b$. This asymmetry means the two sides are **not computing the same function**, so the collapsed solution "everything maps to a single constant" no longer satisfies the optimum — for the predictor-side output to match the un-predicted side, the representation has to carry enough information for $h$ to reconstruct a *direction*, not just a point.
- **(ii) Stop-gradient**: The `stop_grad(z_b)` target makes one branch an "oracle" that the other branch chases. Without stop-gradient, the encoder could minimise the loss by moving $z_b$ and $z_a$ *toward each other* simultaneously — the fastest way being to collapse both to the same constant. With stop-gradient, only the predictor-side pathway receives gradient; $z_b$ is treated as fixed, so the loss can only be decreased by making $h(z_a)$ match a target that the network *cannot currently reshape*. Empirically, removing either the predictor or the stop-gradient causes immediate collapse, so both are load-bearing.

**(c)** Negatives in SimCLR are primarily an **optimisation** device, not an *information* one. The theoretical story (InfoNCE as an MI lower bound) is elegant but the **practical function** of negatives is to inject a repulsive force that keeps the encoder from collapsing. SimSiam shows that an equivalent repulsive force can be obtained from **architectural asymmetry** (predictor head + stop-gradient) without any explicit negatives in the loss. Put differently: the role of negatives is to prevent a degenerate optimum, not to maximise mutual information — any mechanism that achieves the former (negatives, stop-grad asymmetry, DINO's centering + sharpening, Barlow Twins' cross-correlation decorrelation) can produce representations of comparable quality.

**Key points:**
- "Contrastive vs non-contrastive" is about *collapse avoidance mechanisms*, not about whether information is preserved.
- Negatives, stop-grad, centering, decorrelation — all different answers to the same engineering question: "how do I keep the encoder from outputting a constant?"
- This is why the SSL literature has converged: by 2022–2023 contrastive and non-contrastive methods reach near-identical linear-probe accuracy, suggesting the exact mechanism matters less than having **some** anti-collapse term.
</details>

---
## Summary

| Section | Key concept |
|---|---|
| **§0–§1** | SSL = define a pretext task whose label comes for free from $x$; four families — classical pretext / generative-masked / contrastive / non-contrastive |
| **§3 + B1** | **MAE**: mask 75% of ViT patches, encode only visible patches, decode to pixels, **loss on masked positions only** ⇒ no identity shortcut |
| **§4 + B2** | **SimCLR / NT-Xent**: L2-normalise embeddings → $(2B,2B)$ similarity → drop diagonal → temperature-scaled cross-entropy against positive-pair indices |
| **§5** | Masked vs contrastive: different augmentations, different batch-size needs, similar final quality |

**Take-aways:**
- Masked modelling and contrastive learning are the two dominant SSL recipes; they differ in *what* they predict (pixels vs similarities), not in their goal (build a transferable encoder without labels).
- In both paradigms, the **loss design** is the entire trick: MAE's "loss on masked positions only" kills shortcuts; SimCLR's "temperature + normalisation + diagonal mask" turns representation learning into a $(2B-1)$-way classification problem.
- Non-contrastive methods (BYOL / SimSiam / DINO) show that negatives are an *optimisation* device, not an information one — a lesson worth remembering when you read any new SSL paper.